In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import soundfile as sf
import warnings

FRAME_LEN = 2048
Q = 40
STRENGTH = 1.5
EPS = 1e-12


# ---------------- AUDIO ----------------

def load_audio(p):
    x, sr = sf.read(p, dtype="float64")
    if x.ndim > 1:
        x = x[:, 0]
    return x, sr


def save_audio(p, x, sr):
    sf.write(p, x, sr)


# ---------------- TEXT ----------------

def text_to_bits(t):
    return "".join(format(b, "08b") for b in t.encode())


def bits_to_text(b):
    b = b[:len(b)//8*8]
    return "".join(chr(int(b[i:i+8], 2)) for i in range(0, len(b), 8))


def int_to_bits(x):
    return format(x, "032b")


def bits_to_int(b):
    return int(b, 2)


# ---------------- CEPSTRUM ----------------

def cep(frame):
    s = np.fft.fft(frame)
    logm = np.log(np.abs(s) + EPS)
    c = np.fft.ifft(logm).real
    return s, c


def inv_cep(c, phase, n):
    lm = np.fft.fft(c).real
    mag = np.exp(np.clip(lm, -50, 50))
    s = mag * np.exp(1j * phase)
    return np.fft.ifft(s).real[:n]


# ---------------- INDEX ----------------

def idx(n):
    return min(Q, n//2 - 2)


# ---------------- EMBED (STABLE FIX) ----------------

def embed_bit(c, bit):
    c = c.copy()
    i = idx(len(c))

    if bit == "1":
        c[i] += STRENGTH
        c[i+1] -= STRENGTH
    else:
        c[i] -= STRENGTH
        c[i+1] += STRENGTH

    return c


# ---------------- EXTRACT ----------------

def extract_bit(c):
    i = idx(len(c))
    return "1" if c[i] > c[i+1] else "0"


# ---------------- EMBED MESSAGE ----------------

def embed(audio, msg):

    bits = text_to_bits(msg)
    bits = int_to_bits(len(bits)) + bits

    n_frames = len(audio)//FRAME_LEN

    if len(bits) > n_frames:
        warnings.warn("truncated")
        bits = bits[:n_frames]

    out = audio.copy()

    for k in range(len(bits)):
        f = out[k*FRAME_LEN:(k+1)*FRAME_LEN]

        s, c = cep(f)
        ph = np.angle(s)

        c = embed_bit(c, bits[k])

        out[k*FRAME_LEN:(k+1)*FRAME_LEN] = inv_cep(c, ph, FRAME_LEN)

    return out, len(bits)


# ---------------- EXTRACT MESSAGE ----------------

def extract(audio):

    n_frames = len(audio)//FRAME_LEN
    bits = []

    for k in range(n_frames):
        f = audio[k*FRAME_LEN:(k+1)*FRAME_LEN]
        _, c = cep(f)
        bits.append(extract_bit(c))

    bits = "".join(bits)

    length = bits_to_int(bits[:32])
    msg = bits[32:32+length]

    return bits_to_text(msg)


# ---------------- MAIN ----------------

path = input("Enter WAV filename: ")
msg = input("Enter secret message: ")

audio, sr = load_audio(path)

stego, _ = embed(audio, msg)
rec = extract(stego)

save_audio("cepstrum_stego.wav", stego, sr)

print("\nRecovered:", rec)
print("Saved: cepstrum_stego.wav")

In [ ]:
files.download("")

**Observations**

- Embedding and extraction worked successfully.
- Average execution time: 3–5 seconds.
- No significant degradation in audio clarity was observed.
- Cepstrum method performed well for speech-oriented samples.